# Evolution Circuits for Commuting Pauli Strings Relevant to Benchmark Hamiltonians

This notebook computes and stores circuits for unitary evolution under commuting subsets of Pauli strings defining Hamiltonians from the Qiskit **HamLib** repository, producing QASM output compatible with **graphix** patterns.

- Conventions:

    1. Evolution operator

        For Ns terms in the Hamiltonian, H, c_j are string coeffecients written as real numbers for j=0,1,...,Ns and S_j are Pauli strings.

        We can then write **H=c_0 * S_0+c_1 * S_1+...**

        Circuits are evolution of one or more commuting strings with "time" set to c_j.  For example, the one_to_one subseting puts one string per subset.

        Each circuit is then: **Exp[-ic_j * S_j]**

        First order Lie-Trotter decomposition is used for sums of string operators

    2. Transpiling

        The default circuit layout option "none" with transpile optimization 1 uses qiskit v 2.2.3 default layout 

        The optional circuit layout "linear" forces a linear two-qubit gate connectivity


- Code is Structured as: **Imports → Configuration → Helpers → Main** 

- How to use:

    1. Create a fresh environment and install deps:

        python -m venv .venv 

        source .venv/bin/activate  # Windows: .venv\Scripts\activate

        pip install -r requirements.txt



    2. Open and run the notebook:

        jupyter lab


    3. In the Configuration cell, set:


        NAME_TO_FIND = "Be2" (or your target Hamiltonian)

        HAM_INSTANCE_TO_FIND = "ham_JW"

        COLORING_STRATEGY = "one_to_one" (or "smallest_last")

        COUPLING = "none" (or "linear")


    4. Run all cells. Outputs will be saved in:

        qasm_circuit_files_{NAME_TO_FIND}_{COLORING_STRATEGY}/ (per-subset QASM + metadata .py)

        full_circuit_files_{NAME_TO_FIND}/ (full Hamiltonian QASM)

        Stats CSV if enabled.


In [1]:
# Imports
from __future__ import annotations
import logging
import sys
import json
import csv
from pathlib import Path
from typing import Iterable, Dict, Any, List
import logging

import numpy as np
import networkx as nx
import requests
from requests.exceptions import RequestException

# Qiskit imports
import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import QuantumRegister, ClassicalRegister
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter
from qiskit import transpile as transpile_qiskit
from qiskit.circuit import ParameterVector
from qiskit.qasm3 import dumps as dumps_qasm3
from qiskit.transpiler import CouplingMap, Layout

# Configure logging (keeps output visible in console)
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)


In [2]:
# Name of Hamiltonian instance to find in benchpress JSON or local JSON file
NAME_TO_FIND = "Be2"
# Options for NAME_TO_FIND include: 
# For 1-5 qubits: arbitrary1, arbitrary2, arbitrary3, arbitrary4, arbitrary5 (these are all from the nq1to5 json file)
# For 6 or more qubits: tfim, Be2 (6-qubits), H2 (8-qubits), BH (10 qubits), OH (10 and 12 qubits),
# NH (14 qubits), Li2 (14 qubits), C2 (18 qubits), N2 (22 qubits), Na2 (24 qubits)
NQUBITS_FOR_OH = 12  # set only if searching for OH molecule
HAM_INSTANCE_TO_FIND = "ham_JW"
# For For 1-5 qubits arbitrary models: choose 'arbitrary' 
# For molecules choose 'ham_JW'. For tfim choose one of: 
#   graph-1D-grid-pbc-qubitnodes_Lx-16_h-2, graph-1D-grid-pbc-qubitnodes_Lx-26_h-6,
#   graph-2D-grid-nonpbc-qubitnodes_Lx-5_Ly-15_h-3, graph-2D-triag-pbc-qubitnodes_Lx-13_Ly-13_h-0.1,
#   graph-2D-triag-pbc-qubitnodes_Lx-19_Ly-19_h-0.5, graph-2D-triag-nonpbc-qubitnodes_Lx-3_Ly-160_h-0.1,
#   graph-2D-grid-nonpbc-qubitnodes_Lx-19_Ly-19_h-5, graph-2D-grid-pbc-qubitnodes_Lx-4_Ly-148_h-6
COLORING_STRATEGY = "one_to_one"  # 'one_to_one' or 'smallest_last': other strategies available in networkx library, but these two are most relevant for comparison
COUPLING = "none"             # 'none' or 'linear', two different circuit coupling maps, use 'none' for graphs
SAVE_CIRCUIT_TO_FILE = True # set to True to save QASM files to disk
TABULATE_CIRCUIT_STATS = True # set to True to save circuit statistics to CSV file
COMPUTE_FULL_CIRCUIT = True   # also compute/save full Hamiltonian circuit for comparison
PRINT_QASM_TO_STDOUT = False  # set to True to print QASM code to console

# Storage locations
DIR_PATH_STORAGE = Path(f"qasm_circuit_files_{NAME_TO_FIND}_{COLORING_STRATEGY}/")
DIR_PATH_STORAGE.mkdir(parents=True, exist_ok=True)

# Remove circuits that are too large
MAXNUMBER_QUBITS = 593  # 24 is largest JW chemistry Hamiltonian but 592 is largest for tfim in 100_representative.json

# Benchpress and JSON candidates
#check if using arbitrary instance, if so use the nq1to5 json file, 
# otherwise use the 100_representative.json file which includes molecules 
if HAM_INSTANCE_TO_FIND == "arbitrary":
    LOCAL_CANDIDATES = [ Path("pauli_strings_nq1to5.json"), ]
#Use this for molecules 
else: 
    LOCAL_CANDIDATES = [Path("100_representative.json"),]

BENCHPRESS_JSON_URL = (
    "https://raw.githubusercontent.com/Qiskit/benchpress/"
    "e7b29ef7be4cc0d70237b8fdc03edbd698908eff/"
    "benchpress/hamiltonian/hamlib/100_representative.json"
)

# Gateset for graphix
GATESET = ['cx', 's', 'h', 'x', 'y', 'z', 'swap', 'rz']

# Optimization level for transpilation (0-3)
OPT_LEVEL = 1

# Seen for Tranpiler 
SEED_TRANSPILE = 131


In [3]:
# Helpers
def subset_commuting_paulis_minimal(
    sparse_pauli_op: SparsePauliOp,
    coloring_strategy: str = "one_to_one",
) -> List[SparsePauliOp]:
    """
    Partition Pauli terms into unique commuting subsets with minimal number of subsets
    using graph coloring on the anti-commutation graph.

    Args:
        sparse_pauli_op (SparsePauliOp): The input Hamiltonian.
        coloring_strategy (str): 'smallest_last' (greedy) or 'one_to_one' (each term alone).

    Returns:
        List[SparsePauliOp]: A list of SparsePauliOp objects, each containing a commuting subset.
    """
    paulis = sparse_pauli_op.paulis
    coeffs = sparse_pauli_op.coeffs

    # Fast path: one Pauli per subset, in index order
    if coloring_strategy == "one_to_one":
        return [SparsePauliOp([paulis[i]], [coeffs[i]]) for i in range(len(paulis))]

    # Build anti-commutation graph
    G = nx.Graph()
    for i, p1 in enumerate(paulis):
        G.add_node(i)
        for j, p2 in enumerate(paulis):
            if i < j and not p1.commutes(p2):  # edge if they do NOT commute
                G.add_edge(i, j)

    # Graph coloring (greedy heuristic)
    if coloring_strategy == "smallest_last":
        coloring = nx.coloring.greedy_color(G, strategy="smallest_last")
    else:
        raise ValueError(f"coloring strategy {coloring_strategy} not recognized")

    # subset by color deterministically
    color_subsets: Dict[int, List[int]] = {}
    for idx, color in sorted(coloring.items()):
        color_subsets.setdefault(color, []).append(idx)

    subseted_ops: List[SparsePauliOp] = []
    for color in sorted(color_subsets):
        subset = sorted(color_subsets[color])
        sub_paulis = [paulis[i] for i in subset]
        sub_coeffs = [coeffs[i] for i in subset]
        subseted_ops.append(SparsePauliOp(sub_paulis, sub_coeffs))

    return subseted_ops


def load_json_from_local(paths: List[Path]):
    """
    Try loading JSON from the first readable local file found in `paths`.
    Returns (data, message) on success, (None, message) on failure.
    """
    last_error = None
    for p in paths:
        if p.is_file():
            try:
                with p.open("r", encoding="utf-8") as f:
                    data = json.load(f)
                return data, f"Loaded Hamiltonian from local file: {p}"
            except (OSError, json.JSONDecodeError) as e:
                last_error = e
    if last_error is not None:
        return None, f"Found a local file but failed to load/parse JSON: {last_error}"
    return None, "Local Hamiltonian file not found, going to URL."


def load_json_from_url(url: str, timeout: int = 15):
    """
    Download and parse JSON from `url` with a timeout.
    Returns (data, message) on success, (None, message) on failure.
    """
    try:
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        return data, f"Downloaded Hamiltonian from URL: {url}"
    except (RequestException, json.JSONDecodeError) as e:
        return None, f"Failed to download/parse JSON from URL: {e}"


def get_hamiltonian(local_paths: List[Path], url: str):
    """
    a) Search for local file(s) and import if found.
    b) If not, try online.
    c) If neither found, raise a RuntimeError.
    """
    data, msg_local = load_json_from_local(local_paths)
    if data is not None:
        logger.info(msg_local)
        return data

    logger.warning(msg_local)
    data, msg_remote = load_json_from_url(url)
    if data is not None:
        logger.info(msg_remote)
        return data

    logger.error(msg_remote)
    raise RuntimeError("Could not load Hamiltonian JSON from local file(s) or URL.")


def summarize_circuit(expanded) -> Dict[str, Any]:
    s = {
        "name": getattr(expanded, "name", None),
        "instance": HAM_INSTANCE_TO_FIND,
        "num_qubits": expanded.num_qubits,
        "num_clbits": expanded.num_clbits,
        "num_ancillas": expanded.num_ancillas,
        "width": expanded.width(),
        "size": expanded.size(),
        "depth": expanded.depth(),
        "num_parameters": expanded.num_parameters,
        "global_phase": expanded.global_phase,
        "total_instructions": len(expanded.data),
    }
    s["gate_counts"] = dict(expanded.count_ops())
    return s


def circuits_to_csv(circuits: Iterable, csv_path: str):
    rows = [summarize_circuit(c) for c in circuits]
    all_gates = set().union(*[set(r["gate_counts"]) for r in rows]) if rows else set()
    base_cols = [
        "name", "instance", "num_qubits", "num_clbits", "num_ancillas",
        "width", "size", "depth", "num_parameters",
        "global_phase", "total_instructions",
    ]
    gate_cols = [f"gate.{g}" for g in sorted(all_gates)]
    fieldnames = base_cols + gate_cols

    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            flat = {k: v for k, v in r.items() if k != "gate_counts"}
            for g in all_gates:
                flat[f"gate.{g}"] = r["gate_counts"].get(g, 0)
            w.writerow(flat)


def assert_no_ellipsis(path: Path):
    txt = Path(path).read_text(encoding="utf-8")
    if "..." in txt:
        raise RuntimeError(f"Found '...' in {path}; upstream conversion missed something.")




def relabel_cancel_initial_and_final(
    circ: QuantumCircuit,
    original_qubits=None,
    qreg_name: str = "q",
) -> QuantumCircuit:
    """
    Relabel wires so that initial-layout permutation composed with the routing permutation
    becomes the identity on the first `n = len(original_qubits)` wires. No gates are added.
    Any extra (ancilla) qubits are left untouched (identity mapping).

    Returns a NEW circuit whose qubits all live inside a single QuantumRegister named `qreg_name`,
    and whose classical registers are preserved. This guarantees qasm3 export with `qubit[n] q;`.
    """
    # --- sanity on layout / inputs ---
    tl = getattr(circ, "layout", None)
    if tl is None:
        # Nothing to undo; but still ensure a single qreg for clean export.
        return _force_single_register_for_export(circ, qreg_name)

    if original_qubits is None:
        raise ValueError("Provide `original_qubits` (the qubits of the pre-transpile circuit).")

    n = len(original_qubits)
    N = circ.num_qubits

    # 1) initial permutation: input index -> position after initial layout
    #    tl.initial_layout maps Qubit -> position (int)
    init_layout = getattr(tl, "initial_layout", None)
    if init_layout is None:
        # Older/newer versions may expose this differently; fall back to identity
        init_perm = list(range(n))
    else:
        init_perm = [int(init_layout[original_qubits[i]]) for i in range(n)]

    # 2) routing permutation: position before routing -> final position
    #    tl.routing_permutation() returns a list[int] or None
    try:
        rout_perm = tl.routing_permutation()
    except Exception:
        rout_perm = None
    if not rout_perm:
        rout_perm = list(range(N))

    # 3) Compose permutations on the logical subspace [0..n-1]
    #    final_pos = rout_perm[ init_perm[input_pos] ]
    comp = [int(rout_perm[p]) for p in init_perm]

    # If already identity on logical subspace, still normalize registers for export
    if all(i == j for i, j in enumerate(comp)):
        return _force_single_register_for_export(circ, qreg_name)

    # 4) Invert composite on logical subspace: post -> pre
    inv = [0] * n
    for i_pre, i_post in enumerate(comp):
        inv[i_post] = i_pre

    # 5) Build a NEW circuit with explicit registers and explicit bit mapping
    # --- Quantum: single register spanning all qubits (ensures `qubit[N] q;`) ---
    qreg = QuantumRegister(N, qreg_name)
    # --- Classical: preserve the same register structure (names/sizes) if any ---
    if circ.cregs:
        out = QuantumCircuit(qreg, *[ClassicalRegister(reg.size, reg.name) for reg in circ.cregs])
    else:
        # If the circuit has classical bits but no creg objects (rare), put them in one creg.
        out = QuantumCircuit(qreg)
        if circ.num_clbits and not circ.cregs:
            out.add_register(ClassicalRegister(circ.num_clbits, "c"))

    # Preserve phase/metadata
    out.global_phase = circ.global_phase
    out.metadata = getattr(circ, "metadata", None)

    # Build qubit mapping:
    #  - for logical wires (0..n-1): apply inverse permutation -> new index inv[j]
    #  - for ancillas (n..N-1): keep them as-is -> index j
    q_old_to_new = {}
    for j in range(n):
        q_old_to_new[circ.qubits[j]] = out.qubits[inv[j]]
    for j in range(n, N):
        q_old_to_new[circ.qubits[j]] = out.qubits[j]

    # Build classical mapping (1:1 by clbit index in circuit order)
    c_old_to_new = {}
    if circ.num_clbits:
        # After we added cregs to `out` in the same order, their clbits are in the same order too
        for idx, cb in enumerate(circ.clbits):
            c_old_to_new[cb] = out.clbits[idx]

    # 6) Re-append all instructions with remapped operands
    for ci in circ.data:
        inst   = ci.operation
        qargs  = [q_old_to_new[q] for q in ci.qubits]
        cargs  = [c_old_to_new[c] for c in ci.clbits] if ci.clbits else []
        out.append(inst, qargs, cargs)    
    
    return out


def _force_single_register_for_export(circ: QuantumCircuit, qname: str) -> QuantumCircuit:
    """
    Helper: rebuild `circ` onto a single QuantumRegister named `qname`,
    preserving classical registers and all instructions. Useful when no layout is present.
    """
    N = circ.num_qubits
    qreg = QuantumRegister(N, qname)
    if circ.cregs:
        out = QuantumCircuit(qreg, *[ClassicalRegister(reg.size, reg.name) for reg in circ.cregs])
    else:
        out = QuantumCircuit(qreg)
        if circ.num_clbits and not circ.cregs:
            out.add_register(ClassicalRegister(circ.num_clbits, "c"))

    out.global_phase = circ.global_phase
    out.metadata = getattr(circ, "metadata", None)

    # Map qubits 1:1, clbits 1:1
    qmap = {old_q: out.qubits[i] for i, old_q in enumerate(circ.qubits)}
    cmap = {old_c: out.clbits[i] for i, old_c in enumerate(circ.clbits)}
    for ci in circ.data:
        op    = ci.operation
        qargs = [qmap[q] for q in ci.qubits]
        cargs = [cmap[c] for c in ci.clbits] if ci.clbits else []
        out.append(op, qargs, cargs)
    
    return out



In [ ]:
# Main
def main():
    # Validate configuration early
    valid_coupling = {"none", "linear"}
    if COUPLING is None or str(COUPLING).strip() == "":
        raise SystemExit("ERROR: --coupling must be specified as 'none' or 'linear'.")
    coupling_norm = str(COUPLING).strip().lower()
    if coupling_norm not in valid_coupling:
        raise SystemExit(f"ERROR: invalid --coupling {COUPLING!r}. Choose from {sorted(valid_coupling)}.")

    valid_coloring = {"one_to_one", "smallest_last"}
    if COLORING_STRATEGY not in valid_coloring:
        raise SystemExit(f"ERROR: invalid coloring strategy {COLORING_STRATEGY!r}. Choose from {sorted(valid_coloring)}.")

    # Load Hamiltonian records
    ham_records = get_hamiltonian(LOCAL_CANDIDATES, BENCHPRESS_JSON_URL)
    # Filter by qubit count
    ham_records = [h for h in ham_records if h["ham_qubits"] <= MAXNUMBER_QUBITS]

    qc_ham_list = []
    ham_index = None

    # Locate target Hamiltonian
    for h in ham_records:
        name = h["ham_problem"]
        if name == NAME_TO_FIND and HAM_INSTANCE_TO_FIND in h.get("ham_instance", ""):
            if name == "OH" and h["ham_qubits"] != NQUBITS_FOR_OH:
                continue
            terms = h["ham_hamlib_hamiltonian_terms"]
            coeff = h["ham_hamlib_hamiltonian_coefficients"]
            num_qubits = h["ham_qubits"]
            ham_index = len(qc_ham_list)
            qc_ham = QuantumCircuit(num_qubits)
            qc_ham.name = name
            # Keep current behavior: attach attributes to circuit (for compatibility)
            qc_ham.terms = terms
            qc_ham.coeff = coeff
            qc_ham.number_qubits = num_qubits
            qc_ham_list.append(qc_ham)

    if ham_index is None:
        raise SystemExit("ERROR: Target Hamiltonian not found with the given filters.")

    logger.info(f"terms: {qc_ham_list[ham_index].terms}")
    logger.info(f"coeffs: {qc_ham_list[ham_index].coeff}")
    logger.info(f"name: {qc_ham_list[ham_index].name}")
    logger.info(f"number of qubits: {qc_ham_list[ham_index].number_qubits}")
    logger.info(f"number of coeffs: {len(qc_ham_list[ham_index].coeff)}")
    logger.info(f"number of terms: {len(qc_ham_list[ham_index].terms)}")

    nqubits = qc_ham_list[ham_index].number_qubits
    c = ParameterVector("c", length=len(qc_ham_list[ham_index].coeff))

    # Gateset for graphix
    gateset = GATESET

    # Use parametric coeffs
    c_nparray = np.array(c)
    subsets = subset_commuting_paulis_minimal(
        SparsePauliOp(qc_ham_list[ham_index].terms, c_nparray),
        coloring_strategy=COLORING_STRATEGY,
    )

    logger.info(f"Number of subsets: {len(subsets)}")

    # Linear coupling map
    if coupling_norm == "linear":
        line_cmap = CouplingMap.from_line(nqubits, bidirectional=True)

    for gi, subset in enumerate(subsets):
        logger.info(f"paulis: {subset.paulis}")
        pretty = [str(coeff) for coeff in subset.coeffs]
        logger.info(f"coeffs: {pretty}")

    circuits = []
    for subset_index, subset in enumerate(subsets):
        element_sz = len(subset)
        logger.info(f"\n element_sz: {element_sz}")
        qcsubset = QuantumCircuit(nqubits)

        for i, pauli in enumerate(subset.paulis):
            qcsubset.append(
                PauliEvolutionGate(
                    pauli,
                    time=subset.coeffs[i],
                    synthesis=LieTrotter(reps=1, preserve_order=True),
                ),
                range(nqubits),
            )

        # Expand PauliEvolutionGate to standard gates
        qcsubset = qcsubset.decompose(reps=1)

        # Wrap into a larger circuit and transpile 
        qc = QuantumCircuit(nqubits)
        qc.append(qcsubset, range(nqubits))
        init = Layout.from_intlist(list(range(qc.num_qubits)), *qc.qregs)
        if coupling_norm == "linear":
            expanded = transpile_qiskit(
                qc,
                coupling_map=line_cmap,
                initial_layout=init,
                basis_gates=gateset,
                optimization_level=OPT_LEVEL,
                seed_transpiler=SEED_TRANSPILE
            )
            # Relabel wires (no gates appended, only a qubit remap)
            try:
                expanded = relabel_cancel_initial_and_final(expanded, original_qubits=qc.qubits)
            except NameError:
                logger.warning("relabel_cancel_initial_and_final not found; skipped relabeling.")
        else:
            expanded = transpile_qiskit(qc, basis_gates=gateset, optimization_level=OPT_LEVEL, seed_transpiler=SEED_TRANSPILE)

        # Testing no map vs linear map using overlap
        if coupling_norm == "linear":
            vals = {c[i]: coeff[i] for i in range(len(c))}
            # A) 'none' case
            expanded_none = transpile_qiskit(qc, basis_gates=gateset, optimization_level=OPT_LEVEL, seed_transpiler=SEED_TRANSPILE)
            expanded_none_bound = expanded_none.assign_parameters(vals, strict=False)
            sv_none = Statevector.from_instruction(expanded_none_bound)
            # B) 'linear' case
            expanded_lin = transpile_qiskit(
                qc, coupling_map=line_cmap, initial_layout=init, basis_gates=gateset, optimization_level=OPT_LEVEL, seed_transpiler=SEED_TRANSPILE
            )
            try:
                expanded_lin = relabel_cancel_initial_and_final(expanded_lin, original_qubits=qc.qubits)
            except NameError:
                logger.warning("relabel_cancel_initial_and_final not found; skipped relabeling.")
            expanded_lin_bound = expanded_lin.assign_parameters(vals, strict=False)
            sv_lin = Statevector.from_instruction(expanded_lin_bound)
            overlap = abs(sv_lin.data.conj() @ sv_none.data)
            logger.info(f"overlap (linear vs none) = {overlap}")
            if overlap < 1 - 1e-5:
                logger.warning("WARNING: low overlap between linear and none transpiled circuits! Check relabeling.")
                raise SystemExit(1)

        logger.info(f"Number of qubits: {expanded.num_qubits}")
        logger.info(f"Classical bits: {expanded.num_clbits}")
        logger.info(f"Ancillas: {expanded.num_ancillas}")
        logger.info(f"Width: {expanded.width()}")
        logger.info(f"Size (ops): {expanded.size()}")
        logger.info(f"Depth (levels): {expanded.depth()}")
        logger.info(f"Gate counts: {expanded.count_ops()}")
        logger.info(f"Num parameters: {expanded.num_parameters}")
        logger.info(f"Global phase: {expanded.global_phase}")
        logger.info(f"Total instructions: {len(expanded.data)}")

        qasm_str = dumps_qasm3(expanded)
        if SAVE_CIRCUIT_TO_FILE:
            output_qasm_file = DIR_PATH_STORAGE / f"evolution_to_gates_for_graphix_{qc_ham_list[ham_index].name}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}_commsubgrp_{subset_index}.qasm"
            with open(output_qasm_file, "w", encoding="utf-8") as f:
                f.write(qasm_str)
            logger.info(f"\n Wrote QASM code to: {output_qasm_file}")
            assert_no_ellipsis(output_qasm_file)

        if PRINT_QASM_TO_STDOUT:
            print(qasm_str)

        if TABULATE_CIRCUIT_STATS:
            circuits.append(expanded)

    # Full circuit (for comparison)
    if COMPUTE_FULL_CIRCUIT:
        qc_full = QuantumCircuit(nqubits)
        #preserve the order of terms in groupging since terms are non-commuting across subsets, but commute within subsets
        for subset_index, subset in enumerate(subsets):
            for i, pauli in enumerate(subset.paulis):
                qc_full.append(
                    PauliEvolutionGate(
                        pauli,
                        time=subset.coeffs[i],
                        synthesis=LieTrotter(reps=1, preserve_order=True),
                    ),
                    range(nqubits),
                )
        qc_full = qc_full.decompose(reps=1)

        if coupling_norm == "linear":
            init = Layout.from_intlist(list(range(qc_full.num_qubits)), *qc_full.qregs)
            expanded_full = transpile_qiskit(
                qc_full, coupling_map=line_cmap, initial_layout=init, basis_gates=gateset, optimization_level=OPT_LEVEL, seed_transpiler=SEED_TRANSPILE
            )
            try:
                expanded_full = relabel_cancel_initial_and_final(expanded_full, original_qubits=qc_full.qubits)
            except NameError:
                logger.warning("relabel_cancel_initial_and_final not found; skipped relabeling.")
        else:
            expanded_full = transpile_qiskit(qc_full, basis_gates=gateset, optimization_level=OPT_LEVEL, seed_transpiler=SEED_TRANSPILE)

        qasm_str = dumps_qasm3(expanded_full)
        output_qasm_file = DIR_PATH_STORAGE / f"full_evolution_to_gates_for_graphix_{qc_ham_list[ham_index].name}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}.qasm"
        with open(output_qasm_file, "w", encoding="utf-8") as f:
            f.write(qasm_str)
        logger.info(f"\nWrote QASM code for full Hamiltonian circuit to: {output_qasm_file}")
        assert_no_ellipsis(output_qasm_file)  

  
    # Save metadata file (.py)
    if SAVE_CIRCUIT_TO_FILE:
        # --- Environment & tool versions ---
        import platform, hashlib, json
        import numpy as _np
        import networkx as _nx

        output_meta_file = DIR_PATH_STORAGE / f"meta_data_evolution_to_gates_for_graphix_{qc_ham_list[ham_index].name}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}.py"

        # Core inputs already in your metadata
        num_qubits = int(expanded.num_qubits)
        coeff_list = np.asarray(qc_ham_list[ham_index].coeff).tolist()
        terms_list = [str(t) for t in qc_ham_list[ham_index].terms]

        # ---- Extra provenance for reproducibility ----
        # Qiskit/OpenQASM settings and environment
        qiskit_version = getattr(qiskit, "__version__", None)
        python_version = platform.python_version()
        numpy_version = _np.__version__
        networkx_version = _nx.__version__
        qasm_dialect = "openqasm3"  # because dumps_qasm3 was used

        # Transpile configuration (as actually used in this script)
        # If you change these in the transpile(...) calls, update here too.
        optimization_level_used = OPT_LEVEL
        basis_used = list(GATESET) if "GATESET" in locals() else (list(gateset) if "gateset" in locals() else None)

        # Target/backend and coupling info
        # You used no backend/target; for 'linear' we record the coupling edges of the line map.
        target_or_backend = None

        # Attempt to capture the transpiler's initial layout (string repr, if available)
        initial_layout_repr = None
        try:
            tl = getattr(expanded, "layout", None)
            if tl is not None and hasattr(tl, "initial_layout"):
                # store a string representation to avoid object serialization headaches
                initial_layout_repr = str(tl.initial_layout)
        except Exception:
            initial_layout_repr = None

        # Methods (only fill in if you explicitly set them elsewhere)
        routing_method = None
        layout_method = None
        translation_method = None
        seed_transpiler = SEED_TRANSPILE 

        transpile_config = {
            "optimization_level": optimization_level_used,
            "basis_gates": basis_used,
            "target_or_backend": target_or_backend,
            "initial_layout": initial_layout_repr,
            "routing_method": routing_method,
            "layout_method": layout_method,
            "translation_method": translation_method,
            "seed_transpiler": seed_transpiler,
        }

        # Synthesis/decomposition details used in your construction
        synthesis_config = {"method": "LieTrotter", "reps": 1, "preserve_order": True}
        decompose_reps = 1

        # Grouping info (deterministic ordering via NetworkX greedy_color with "smallest_last")
        grouping_info = {
            "algorithm": "networkx.greedy_color(strategy='smallest_last')",
            "coloring_strategy": COLORING_STRATEGY,
            "networkx_version": networkx_version,
        }

        # Benchpress source + content checksum (slice you consumed)
        # If you loaded locally, BENCHPRESS_JSON_URL may still point to the canonical URL.
        try:
            benchpress_source_url = BENCHPRESS_JSON_URL
        except NameError:
            benchpress_source_url = None

        ham_slice = {
            "ham_problem": qc_ham_list[ham_index].name,
            "ham_instance": HAM_INSTANCE_TO_FIND,
            "terms": terms_list,
            "coefficients": coeff_list,
        }
        ham_sha256 = hashlib.sha256(json.dumps(ham_slice, sort_keys=True).encode("utf-8")).hexdigest()

        # Circuit stats snapshot (redundant with CSV, but handy to collocate)
        gate_counts = dict(expanded.count_ops())
        num_parameters = int(expanded.num_parameters)
        global_phase = expanded.global_phase
        total_instructions = len(expanded.data)

        with open(output_meta_file, "w", encoding="utf-8") as f:
            # Original header
            f.write(f"#Metadata_for_circuit_evolution_to_gates_for_graphix_{qc_ham_list[ham_index].name}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}\n")
            f.write(f"Number_qubits= {num_qubits}\n")
            f.write(f"Coefficients= {coeff_list}\n")
            f.write(f"Terms= {terms_list}\n")
            f.write(f"Coloring_strategy= '{COLORING_STRATEGY}'\n")
            f.write(f"circuit_mapping= '{COUPLING}'\n")
            f.write(f"Hamiltonian_Instance = '{HAM_INSTANCE_TO_FIND}'\n")

            # ---- New fields: SDK/env versions ----
            f.write(f"qiskit_version = '{qiskit_version}'\n")
            f.write(f"python_version = '{python_version}'\n")
            f.write(f"numpy_version = '{numpy_version}'\n")
            f.write(f"networkx_version = '{networkx_version}'\n")
            f.write(f"qasm_dialect = '{qasm_dialect}'\n")

            # ---- New fields: transpiler configuration ----
            f.write(f"Transpile_config = {transpile_config}\n")
            f.write(f"Seed_transpiler = {SEED_TRANSPILE}\n")

            # ---- New fields: synthesis/decomposition ----
            f.write(f"Synthesis_config = {synthesis_config}\n")
            f.write(f"Decompose_reps = {decompose_reps}\n")

            # ---- New fields: grouping info ----
            f.write(f"Grouping_info = {grouping_info}\n")

            # ---- New fields: source and checksum ----
            f.write(f"Benchpress_source = {{'url': {repr(benchpress_source_url)}, 'sha256_slice': '{ham_sha256}'}}\n")

            # ---- New fields: circuit stats ----
            f.write(f"Gate_counts = {gate_counts}\n")
            f.write(f"Num_parameters = {num_parameters}\n")
            f.write(f"Global_phase = {global_phase}\n")
            f.write(f"Total_instructions = {total_instructions}\n")

            # Existing commuting subset dumps (unchanged)
            for gi, subset in enumerate(subsets):
                labels = list(subset.paulis.to_labels())
                f.write(f"\n# Paulis_for_commuting_subset {gi}\n")
                f.write(f"commuting_paulis_{gi} = {labels}\n")

    assert_no_ellipsis(output_meta_file)

    if TABULATE_CIRCUIT_STATS:
        circuits_to_csv(
            circuits, DIR_PATH_STORAGE / f"circuit_stats_{qc_ham_list[ham_index].name}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}.csv",
        )


if __name__ == "__main__":
    # In a notebook, this will run when the cell executes.
    try:
        main()
    except RuntimeError as e:
        logger.error(str(e))

[INFO] Loaded Hamiltonian from local file: 100_representative.json
[INFO] terms: ['IIIIII', 'XXYYII', 'XXIIYY', 'XYYXII', 'XYIIYX', 'XZXXZX', 'XZXYZY', 'XZZZXI', 'XZZZXZ', 'XZZIXI', 'XZIZXI', 'XIZZXI', 'YXXYII', 'YXIIXY', 'YYXXII', 'YYIIXX', 'YZYXZX', 'YZYYZY', 'YZZZYI', 'YZZZYZ', 'YZZIYI', 'YZIZYI', 'YIZZYI', 'ZIIIII', 'ZXZZZX', 'ZYZZZY', 'ZZIIII', 'ZIZIII', 'ZIIZII', 'ZIIIZI', 'ZIIIIZ', 'IXXYYI', 'IXYYXI', 'IXZZZX', 'IXZZIX', 'IXZIZX', 'IXIZZX', 'IYXXYI', 'IYYXXI', 'IYZZZY', 'IYZZIY', 'IYZIZY', 'IYIZZY', 'IZIIII', 'IZZIII', 'IZIZII', 'IZIIZI', 'IZIIIZ', 'IIXXYY', 'IIXYYX', 'IIYXXY', 'IIYYXX', 'IIZIII', 'IIZZII', 'IIZIZI', 'IIZIIZ', 'IIIZII', 'IIIZZI', 'IIIZIZ', 'IIIIZI', 'IIIIZZ', 'IIIIIZ']
[INFO] coeffs: [-28.48651135482841, -0.01574633931772222, -0.010562347732704107, 0.01574633931772222, 0.010562347732704107, 0.011922473641009169, 0.011922473641009169, 0.024660238369096588, 0.00391847450165415, 0.0012622097443250315, 0.013184683385334201, 0.014131831698376306, 0.01574633931772222,